In [ ]:
!pip install -U langchain langgraph langchain_anthropic

In [1]:
import asyncio
import operator
from typing import Annotated, TypedDict

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import END, StateGraph
from engram.sdk.client import Engram
class State(TypedDict):
    messages: Annotated[list, operator.add]   # Chat messages — appended by each node
    context_id: str                           # Which Engram context to use
    materialization_id: str | None            # Receipt from last recall (for reconsolidation)


async def recall(state: State) -> dict:
    last_msg = state["messages"][-1].content     # The user's message

    mat = await engram.materialize(
        context_id=state["context_id"],           # Which knowledge graph to search
        query=last_msg,                           # Semantic search using this text
        token_budget=2000,                        # Max tokens to return
    )

    # Inject memories as a system message. Because messages uses
    # operator.add, this gets APPENDED — the user's message stays.
    # Claude will see: [HumanMessage, SystemMessage(memories)]
    system = SystemMessage(content=f"Your memory:\n{mat['rendered_text']}")

    return {
        "messages": [system],                     # Appended to messages list
        "materialization_id": mat["materialization_id"],
            # Receipt: "these specific bullets were recalled."
            # Passed to commit for reconsolidation feedback.
    }


async def respond(state: State) -> dict:
    response = await llm.ainvoke(state["messages"])  # Claude generates response
    return {"messages": [response]}                  # AIMessage appended to list


In [2]:
CONTEXT_ID = None  # Will be created on first run, reused after

llm = ChatAnthropic(model="claude-sonnet-4-20250514")  # Agent's LLM
engram = Engram(url="http://localhost:5820")            # Memory server


async def setup_context(state: State) -> dict:
    global CONTEXT_ID
    if CONTEXT_ID is None:
        ctx = await engram.create_context(
            name="Research Assistant",
            intent={
                "objective": "Help user with research tasks",
                "success_criteria": ["Remember key findings across sessions"],
            },
        )
        CONTEXT_ID = str(ctx.id)
    return {"context_id": CONTEXT_ID}



async def commit(state: State) -> dict:
    msgs = state["messages"]
    human = [m for m in msgs if isinstance(m, HumanMessage)][-1]
    ai = [m for m in msgs if isinstance(m, AIMessage)][-1]

    await engram.commit(
        context_id=state["context_id"],           # Which knowledge graph to write to
        agent_id="langgraph-research",            # Identifies this agent for provenance.
                                                  # Shows up in the activity ledger and
                                                  # agent dashboard. Also used for event
                                                  # notifications in multi-agent setups.
        content=f"User: {human.content}\nAssistant: {ai.content}",
                                                  # Raw text — stored permanently AND
                                                  # processed by the Reflector into bullets.
                                                  # This is the "source code" that can be
                                                  # re-extracted with a better model later.
        content_type="conversation",              # Tells the Reflector what extraction
                                                  # strategy to use:
                                                  #   "conversation" → extract insights,
                                                  #     decisions, strategies from dialogue
                                                  #   "tool_output" → extract success/fail
                                                  #     signals, metrics, error patterns
                                                  #   "document" → extract facts, procedures,
                                                  #     constraints from structured text
        materialization_id=state.get("materialization_id"),
                                                  # Links to the previous recall. This is
                                                  # what enables reconsolidation:
                                                  #   "The bullets you gave me in that
                                                  #    recall? They helped (or didn't)."
                                                  # If None (no prior recall), reconsolidation
                                                  # is skipped but knowledge is still saved.
        source_model="claude-sonnet-4",           # Which LLM generated this text.
                                                  # Metadata only — does NOT affect which
                                                  # model processes the extraction. That's
                                                  # always the server's canonical Reflector.
    )
    return {}

graph = StateGraph(State)
graph.add_node("setup", setup_context)
graph.add_node("recall", recall)
graph.add_node("respond", respond)
graph.add_node("commit", commit)

graph.set_entry_point("setup")
graph.add_edge("setup", "recall")
graph.add_edge("recall", "respond")
graph.add_edge("respond", "commit")
graph.add_edge("commit", END)

app = graph.compile()


async def chat(message: str):
    result = await app.ainvoke({
        "messages": [HumanMessage(content=message)],  # Fresh message list each time
        "materialization_id": None,                   # No prior recall yet
    })
    return result["messages"][-1].content             # Return Claude's response




In [3]:
await chat("I recall there are many types of Chinese dumplings when I grew up: northeastern style with porn and chives, which is very traditional, beijing style, guangdong style with shrimp, yunnan style, eastern styles and Sichuan style from my home province")

'What a wonderful collection of dumpling memories from your childhood! It sounds like you experienced quite a diverse range of regional Chinese dumpling styles. I can see your deep connection to these foods, especially noting that Sichuan is your home province.\n\nJust to clarify - when you mention "northeastern style with porn and chives," I believe you might have meant "pork and chives"? That\'s indeed a very classic and traditional filling combination that\'s especially popular in northeastern China.\n\nYour list covers such a beautiful geographic spread:\n- **Northeastern** (dongbei) - often heartier, with that classic pork and chive combination\n- **Beijing style** - traditionally more refined, sometimes featuring lamb\n- **Guangdong style** - with delicate shrimp har gow being a dim sum staple\n- **Yunnan style** - often unique with local ingredients and sometimes different wrapper styles\n- **Eastern styles** - likely including Shanghai\'s soup dumplings (xiaolongbao)\n- **Sichu

In [4]:
await chat("I love all of them. But my favorites are steamed dumpling (shanghai stayle) and dry dumplings with chilli oil (chengdu/sichuan style) . Well, if I have pick one, I guess it would be Chengdu, which reminds me of my home town.")

"That's wonderful that Chengdu-style dumplings connect you to your hometown! There's something so special about how certain dishes can instantly transport us back home through taste and memory.\n\nChengdu's dry dumplings with chili oil (often called *hong you chao shou* 红油抄手) are absolutely incredible - that perfect balance of the silky dumpling wrappers, savory filling, and that distinctive Sichuan mala flavor from the chili oil and Sichuan peppercorns. The way the oil coats each dumpling with layers of heat and numbing spice is truly an art form.\n\nAnd Shanghai-style steamed dumplings are such a beautiful contrast! I'm guessing you might be thinking of *xiaolongbao* 小笼包 with their delicate pleated tops and that amazing burst of soup broth inside? Or perhaps the slightly larger steamed dumplings that are more common there?\n\nIt sounds like you have a wonderful appreciation for both the comfort of home flavors and the refined techniques of different regions. Do you find yourself maki

In [5]:
CONTEXT_ID

'20102edd-76ca-4a3c-88ed-1b9721f8ceb4'

In [ ]:
CONTEXT_ID = '20102edd-76ca-4a3c-88ed-1b9721f8ceb4'
llm = ChatAnthropic(model="claude-sonnet-4-20250514")  # Agent's LLM
engram = Engram(url="http://localhost:5820")            # Memory server


# Read-only graph: recall → respond (no setup, no commit)
read_graph = StateGraph(State)
read_graph.add_node("recall", recall)
read_graph.add_node("respond", respond)

read_graph.set_entry_point("recall")
read_graph.add_edge("recall", "respond")
read_graph.add_edge("respond", END)

read_app = read_graph.compile()

async def ask(message: str):
    result = await read_app.ainvoke({
        "messages": [HumanMessage(content=message)],
        "context_id": CONTEXT_ID,
        "materialization_id": None,
    })
    return result["messages"][-1].content

await ask("What is my favorite dumpling style if you recall?")

"Based on our previous conversation, your favorite dumpling style is **Chengdu-style dry dumplings with chili oil** - which makes perfect sense given your hometown connection to Chengdu! \n\nThese are likely either *zhong shui jiao* (钟水饺) or *hong you chao shou* (红油抄手), both featuring that distinctive Sichuan málà flavor with the spicy, numbing chili oil that makes them so special. It's wonderful how your food preferences align so naturally with your roots - there's something deeply satisfying about that connection between hometown flavors and personal taste.\n\nDo you have any particular memories of enjoying these dumplings back in Chengdu? Maybe a favorite street vendor or family preparation style that made them extra special for you?"